# Phase 4: Predictive Analysis

## Hypothesis: 
Future glucose (e.g., 30–60 minutes out) can be predicted more accurately by combining recent glucose trends with recent physical activity and ingested insulin/carbs, rather than looking at glucose trends alone.

## Model Selection: 
While LSTMs are great for time series, XGBoost or LightGBM are vastly superior for hackathons. They train instantly, handle tabular lag features beautifully, and provide feature importance out-of-the-box.

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def train_predictive_model(df):
    df['glucose_in_30m'] = df['glucose'].shift(-6)
    df['glucose_in_60m'] = df['glucose'].shift(-12)
    
    # Calculate a rolling sum of active insulin/carbs over the last 2 hours
    df['carbs_last_2h'] = df['carb_input'].rolling(window=24, min_periods=1).sum()
    df['bolus_last_2h'] = df['bolus_volume_delivered'].rolling(window=24, min_periods=1).sum()
    df['steps_last_2h'] = df['steps'].rolling(window=24, min_periods=1).sum()
    
    # Feature Engineering: Lags and Rolling Windows
    df['glucose_lag_1'] = df['glucose'].shift(1) # 5 mins ago
    df['glucose_lag_3'] = df['glucose'].shift(3) # 15 mins ago
    df['glucose_rate_of_change'] = df['glucose'] - df['glucose_lag_3']
    
    # Target: Predict glucose 45 minutes in the future (9 steps * 5 mins)
    df['target_glucose_45m'] = df['glucose'].shift(-9)
    
    # Drop NaNs created by shifting
    model_df = df.dropna()
    
    features = ['glucose', 'glucose_lag_1', 'glucose_lag_3', 'glucose_rate_of_change', 
                'carbs_last_2h', 'bolus_last_2h', 'steps_last_2h', 'heart_rate']
    
    X = model_df[features]
    y = model_df['target_glucose_45m']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False) # MUST be False for time series
    
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    
    print(f"Mean Absolute Error predicting 45 mins ahead: {mae:.2f} mg/dL")
    return model, features

In [2]:
import import_ipynb
from Data_Preprocessing_Cleaning import read_and_preprocess_data

In [3]:
cleaned_df = read_and_preprocess_data('/Users/jputha177@cable.comcast.com/Downloads/HUPA0002P.csv')

In [4]:
model, features = train_predictive_model(cleaned_df)

Mean Absolute Error predicting 45 mins ahead: 17.22 mg/dL


In [5]:
features

['glucose',
 'glucose_lag_1',
 'glucose_lag_3',
 'glucose_rate_of_change',
 'carbs_last_2h',
 'bolus_last_2h',
 'steps_last_2h',
 'heart_rate']

In [6]:
df = cleaned_df.copy()
df['glucose_in_30m'] = df['glucose'].shift(-6)
df['glucose_in_60m'] = df['glucose'].shift(-12)

# Calculate a rolling sum of active insulin/carbs over the last 2 hours
df['carbs_last_2h'] = df['carb_input'].rolling(window=24, min_periods=1).sum()
df['bolus_last_2h'] = df['bolus_volume_delivered'].rolling(window=24, min_periods=1).sum()
df['steps_last_2h'] = df['steps'].rolling(window=24, min_periods=1).sum()

# Feature Engineering: Lags and Rolling Windows
df['glucose_lag_1'] = df['glucose'].shift(1) # 5 mins ago
df['glucose_lag_3'] = df['glucose'].shift(3) # 15 mins ago
df['glucose_rate_of_change'] = df['glucose'] - df['glucose_lag_3']

# Target: Predict glucose 45 minutes in the future (9 steps * 5 mins)
df['target_glucose_45m'] = df['glucose'].shift(-9)

# Drop NaNs created by shifting
model_df = df.dropna()

features = ['glucose', 'glucose_lag_1', 'glucose_lag_3', 'glucose_rate_of_change', 
            'carbs_last_2h', 'bolus_last_2h', 'steps_last_2h', 'heart_rate']

X = model_df[features]
y = model_df['target_glucose_45m']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False) # MUST be False for time series

model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)

print(f"Mean Absolute Error predicting 45 mins ahead: {mae:.2f} mg/dL")


Mean Absolute Error predicting 45 mins ahead: 17.22 mg/dL


In [7]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_predictive_model(model, X_test, y_test, feature_names):
    # 1. Generate predictions
    predictions = model.predict(X_test)
    
    # 2. Calculate core metrics
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    
    print(f"--- Model Performance Metrics ---")
    print(f"Mean Absolute Error (MAE): {mae:.2f} mg/dL")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f} mg/dL\n")
    
    # 3. Plot 1: True vs Predicted Over Time (Time Series View)
    # We slice the first 300 data points (~25 hours) so the chart isn't too cluttered
    plt.figure(figsize=(14, 5))
    plt.plot(y_test.values[:300], label='Actual Glucose (Future)', color='#1f77b4', linewidth=2)
    plt.plot(predictions[:300], label='Predicted Glucose', color='#ff7f0e', linestyle='--', linewidth=2)
    
    # Add clinical threshold lines
    plt.axhline(y=70, color='red', alpha=0.5, linestyle=':', label='Hypo Threshold (70)')
    plt.axhline(y=180, color='orange', alpha=0.5, linestyle=':', label='Hyper Threshold (180)')
    
    plt.title('Model Forecasting: 45 Minutes Ahead')
    plt.xlabel('Time Steps (5-min intervals)')
    plt.ylabel('Glucose (mg/dL)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # 4. Plot 2: Feature Importance (Why the model made its choice)
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    sorted_features = [feature_names[i] for i in indices]
    
    plt.figure(figsize=(10, 5))
    plt.title("What drove the prediction? (Feature Importances)")
    plt.bar(range(len(feature_names)), importances[indices], align="center", color='#2ca02c')
    plt.xticks(range(len(feature_names)), sorted_features, rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    return predictions

In [ ]:
evaluate_predictive_model(model, X_test, y_test, features)

--- Model Performance Metrics ---
Mean Absolute Error (MAE): 17.22 mg/dL
Root Mean Squared Error (RMSE): 24.35 mg/dL

